# Benchmark: vLLM vs Ollama

This notebook measures inference latency (Time To First Token) and throughput (Tokens Per Second) between Local Ollama and vLLM on a T4 GPU.

## 1. Install Dependencies & Download Models

In [ ]:
!pip install vllm openai requests
!apt-get update && apt-get install -y zstd\n\n
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh


## 2. Start Ollama Server (Background)

In [ ]:
import subprocess
import time

# Start Ollama in the background
ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(3) # Wait for it to start

# Pull the model
!ollama pull llama3.1

## 3. Start vLLM Server (Background)

In [ ]:
# Start vLLM in the background
vllm_process = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server", 
    "--model", "meta-llama/Meta-Llama-3.1-8B-Instruct", 
    "--port", "8000",
    "--gpu-memory-utilization", "0.8",
    "--max-model-len", "4096"
])

print("Waiting 60 seconds for vLLM to download weights and start up...")
time.sleep(60)

## 4. Run Benchmark

In [ ]:
import requests
import time
import statistics

VLLM_URL = "http://localhost:8000/v1/chat/completions"
OLLAMA_URL = "http://localhost:11434/api/generate"

PROMPT = "Explain the importance of the Socratic method in education."
NUM_REQUESTS = 5

def benchmark_ollama():
    print(f"\n--- Benchmarking Ollama (Model: llama3.1:latest) ---")
    payload = {
        "model": "llama3.1:latest",
        "prompt": PROMPT,
        "stream": False,
        "options": {"temperature": 0.0}
    }
    
    latencies = []
    tokens_per_second = []
    
    for i in range(NUM_REQUESTS):
        print(f"Request {i+1}/{NUM_REQUESTS}...")
        start_time = time.time()
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
        end_time = time.time()
        
        if response.status_code == 200:
            data = response.json()
            latency = end_time - start_time
            latencies.append(latency)
            
            if "eval_count" in data and "eval_duration" in data:
                tps = data["eval_count"] / (data["eval_duration"] / 1e9)
                tokens_per_second.append(tps)
            else:
                words = len(data.get("response", "").split())
                tokens_per_second.append((words * 1.3) / latency)
        else:
            print(f"Error: {response.status_code}")
            
    if latencies:
        print(f"Average Latency: {statistics.mean(latencies):.2f} seconds")
        print(f"Average Throughput: {statistics.mean(tokens_per_second):.2f} tokens/second")

def benchmark_vllm():
    print(f"\n--- Benchmarking vLLM (Model: meta-llama/Meta-Llama-3.1-8B-Instruct) ---")
    payload = {
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "messages": [{"role": "user", "content": PROMPT}],
        "stream": False,
        "temperature": 0.0
    }
    
    headers = {"Authorization": "Bearer EMPTY"}
    latencies = []
    tokens_per_second = []
    
    for i in range(NUM_REQUESTS):
        print(f"Request {i+1}/{NUM_REQUESTS}...")
        start_time = time.time()
        response = requests.post(VLLM_URL, json=payload, headers=headers, timeout=120)
        end_time = time.time()
        
        if response.status_code == 200:
            data = response.json()
            latency = end_time - start_time
            latencies.append(latency)
            
            usage = data.get("usage", {})
            completion_tokens = usage.get("completion_tokens", 0)
            if completion_tokens > 0:
                tps = completion_tokens / latency
                tokens_per_second.append(tps)
        else:
            print(f"Error: {response.status_code} - {response.text}")

    if latencies:
        print(f"Average Latency: {statistics.mean(latencies):.2f} seconds")
        print(f"Average Throughput: {statistics.mean(tokens_per_second):.2f} tokens/second")

benchmark_ollama()
benchmark_vllm()

## 5. Cleanup

In [ ]:
ollama_process.terminate()
vllm_process.terminate()